<a href="https://colab.research.google.com/github/elenasabbioni/SMARTbiomed_summerschool_2026/blob/main/LDSC_Practical_1_h2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astheeggeggs/testing_colab/blob/main/LDSC_Practical_1_h2.ipynb)

# 🧬 Practical 1: SNP Heritability estimation using LD Score Regression (LDSC)

Welcome to the first practical! Here, we will learn how to estimate the SNP heritability ($h^2_{\text{SNP}}$) of a trait using LD Score Regression (LDSC).


---
## 🛠️ 1. Environment Setup & Package Installation

Before we get cracking on the practical, we need to get our environment set up, and all the required R packages installed.

This Colab notebook uses a **Python runtime**, but since the LDSC tools we are using are built in R, we will execute our code using **R cell magic** (`%%R`). We first need to load this extension and install the required packages from CRAN and GitHub.

We'll also copy across the data files required for this practical using the package `gdown` which lets us copy across from a public google drive link.

Packages used in this session:
* `data.table` & `dplyr`: For fast and efficient data manipulation.
* `remotes`: To install packages directly from GitHub.
* `GenomicSEM`: An R package that includes functions to run LDSC.


In [1]:
# System dependencies that are often useful for R package installation
# (safe to run even if some packages are already present)
!apt-get update -qq
!apt-get install -y -qq libcurl4-openssl-dev libssl-dev libxml2-dev
!python -m pip -q install gdown
!gdown --folder "https://drive.google.com/drive/folders/1BIBaoLlodLG_ni9DASh_bw-3RzUZSSJe?usp=sharing" -O /content/LDSC_Practical_1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libssl-dev_3.0.2-0ubuntu1.25_amd64.deb ...
Unpacking libssl-dev:amd64 (3.0.2-0ubuntu1.25) over (3.0.2-0ubuntu1.23) ...
Preparing to unpack .../libssl3_3.0.2-0ubuntu1.25_amd64.deb ...
Unpacking libssl3:amd64 (3.0.2-0ubuntu1.25) over (3.0.2-0ubuntu1.23) ...
Setting up libssl3:amd64 (3.0.2-0ubuntu1.25) ...
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libxml2-dev_2.9.13+dfsg-1ubuntu0.12_amd64.deb ...
Unpacking libxml2-dev:amd64 (2.9.13+dfsg-1ubuntu0.12) over (2.9.13+dfsg-1ubuntu0.11) ...
Preparing to unpack .../libxml2_2.9.13+dfsg-1ubuntu0.12_amd64.deb ...
Unpacking libxml2:amd64 (2.9.13+dfsg-1ubuntu0.12) over (2.9.13+dfsg-1ubuntu

In [2]:
# Load the R magic extension to run R code in this Python notebook
%load_ext rpy2.ipython

In [3]:
%%R
# Install CRAN packages
install.packages(
  c("data.table", "dplyr", "remotes"),
  repos = "https://cloud.r-project.org",
  quiet = TRUE
)

# Install GenomicSEM from GitHub
# GenomicSEM is maintained on GitHub and its README recommends installation from GitHub.
remotes::install_github("GenomicSEM/GenomicSEM", upgrade = "never",
                        quiet = TRUE)


Installing 22 packages: R.methodsS3, iterators, colorspace, gridBase, sfsmisc, R.oo, foreach, quadprog, numDeriv, pbivnorm, mnormt, gtools, proxy, simsalapar, mgsub, R.utils, splitstackshape, doParallel, lavaan, gdata, e1071, plyr


In [4]:
%%R
dir("/content/LDSC_Practical_1")
setwd("/content/LDSC_Practical_1")

### ✅ Load Required Packages

If the installation cell ran successfully, let's load our libraries!


In [5]:
%%R
suppressWarnings(suppressPackageStartupMessages({
  library(data.table)
  library(dplyr)
  library(GenomicSEM)
}))


---
## 📚 2. Understanding LDSC Basics

We are using the `GenomicSEM` library in R to run LDSC. There are **two primary steps** involved in running an LDSC analysis:

1.   **Munge the summary statistics:** `munge()`
     * *To "munge" means to convert raw data from one format to another, cleaning and standardizing it.*
2.   **Run LD Score Regression:** `ldsc()`
     * *Using the cleaned data to estimate heritability and assess confounding.*


### 🧠 Intuition check: What *is* an LD score?

Before we run anything: the **LD score** of a SNP $j$ is defined as
$$\ell_j = \sum_{k} r^2_{jk},$$
the sum of squared correlations ($r^2$) between SNP $j$ and every other SNP $k$ (including itself). If SNP $A$ has $\ell_A = 200$ and SNP $B$ has $\ell_B = 2$, what does that tell you about them?

<details>
<summary>💡 Click here to reveal the answer!</summary>

The LD score measures **how much of the genome a SNP "tags"** through linkage disequilibrium. SNP $A$ ($\ell_A = 200$) sits in a dense LD region and is correlated with many neighbours; SNP $B$ ($\ell_B = 2$) is nearly "lonely" — it tags little beyond itself.

LD scores are **pre-computed** from a reference panel (that's the `eur_w_ld_chr/` directory). Crucially they must come from the **same ancestry** as your GWAS, because LD patterns differ between populations.
</details>


### 🧠 Intuition check: Why do high-LD SNPs tend to have larger $\chi^2$?

Under a **polygenic** trait (many causal variants scattered across the genome), would you expect a SNP that tags *many* other variants to have a larger or smaller association statistic ($\chi^2$) on average — and why?

<details>
<summary>💡 Click here to reveal the answer!</summary>

**Larger, on average.** The more variants a SNP tags, the more likely it is to be in LD with *some* causal variant, so it "borrows" association signal from its neighbours. Across the genome this produces a **positive relationship between a SNP's LD score $\ell_j$ and its expected $\chi^2$** — and the *steepness* of that line is what LDSC turns into a heritability estimate. In one sentence:

> *Do SNPs that tag more of the genome have larger marginal effect sizes?*

Contrast this with **confounding** (e.g. population stratification), which inflates $\chi^2$ for *all* SNPs roughly equally — regardless of LD score.
</details>


### 🧠 Intuition check: The LDSC regression model

LDSC regresses each SNP's $\chi^2$ on its LD score. In expectation,
$$E[\chi^2_j] = \underbrace{\frac{N h^2}{M}}_{\text{slope}} \, \ell_j \; + \; \underbrace{1 + N a}_{\text{intercept}},$$
where $N$ = sample size, $M$ = number of SNPs, $h^2$ = SNP heritability, $a$ = confounding. Which part carries the heritability, and which the confounding?

<details>
<summary>💡 Click here to reveal the answer!</summary>

* **The slope** $\frac{N h^2}{M}$ multiplies $\ell_j$ and is **proportional to $h^2$**. A steeper slope ⇒ more SNP heritability. (Given $N$ and $M$, LDSC backs $h^2$ out of the fitted slope.)
* **The intercept** $1 + Na$ is the part that does **not** depend on LD score. With no confounding ($a = 0$) it sits at **1**; an intercept meaningfully above 1 flags **confounding** (population stratification, sample overlap, …).

So the single picture of "$\chi^2$ vs LD score" separates **real polygenic signal** (slope) from **confounding** (intercept) — that separation is the whole point of LDSC.
</details>


### 🧠 Intuition check: Predict before you run

You're about to estimate $h^2_{\text{SNP}}$ for **BMI** from a large, well-controlled GWAS. *Before* you look at the output: do you expect the LDSC **intercept** to be close to 1, or much greater than 1? Why?

<details>
<summary>💡 Click here to reveal the answer!</summary>

For a large, carefully QC'd GWAS with good ancestry control, expect the intercept to be **close to 1** — there shouldn't be much confounding, so almost all of the inflation in the test statistics should come from **true polygenic signal** (the slope), not stratification. Hold onto this prediction and check it against **Question 10**.
</details>


### 📝 Required Information in Summary Statistics
The input summary statistics file for the `munge()` function must contain at least the following five pieces of information:

1.   **SNP ID:** The rsID of the SNP.
2.   **A1:** The effect allele.
3.   **A2:** The other allele.
4.   **Z-score:** The direction (sign) and magnitude of the association (or effect size and standard error, or $P$-value and effect direction).
5.   **N:** The sample size for the SNP.


### ⚙️ Arguments for the `munge()` Function
The `munge()` function takes several key arguments:
1. `files`: The path/name of the summary statistics file.
2. `hm3`: The path/name of the reference file (e.g., HapMap 3 SNP list).
3. `trait.names`: A string specifying the output prefix/name.
4. `info.filter`: An imputation quality score cutoff (e.g., `0.90`).
5. `maf.filter`: A minor allele frequency cutoff (e.g., `0.01`).


### ⚙️ Arguments for the `ldsc()` Function
The `ldsc()` function requires:
1.   `traits`: A vector of file paths pointing to the munged sumstats.
2.   `sample.prev`: Sample prevalence of the trait (use `NA` for continuous traits or if unknown).
3.   `population.prev`: Population prevalence of the trait (use `NA` for continuous traits or if unknown).
4.   `ld`: The directory containing the pre-computed LD scores.
5.   `wld`: The directory containing the LD scores used for weights (often the same as `ld`).
6.   `trait.names`: A character vector of trait names.


---
## 📁 3. Directory Structure & Data Preparation

The code below assumes you have a folder structured like this:

```text
LDSC_Practical_1/
  EUR/
    eur_w_ld_chr/
      w_hm3.snplist
    CHR22only_Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz
    Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz
    Munged_BMI_Meta-analysis_Locke_et_al+UKBiobank_2018_for_LDSC.sumstats.gz
    BMI.sumstats.gz
```


In [6]:
%%R
# Set directories
sumstat_dir  <- "EUR/"
ref_data_dir <- "EUR/eur_w_ld_chr/"


### 🌍 Reference Data: HapMap 3 (HM3) SNPs
The reference directory includes the HapMap 3 (HM3) SNP IDs and their pre-computed LD scores.


In [7]:
%%R
# Quick check
getwd()
dir(ref_data_dir)


 [1] "1.l2.ldscore.gz"     "1.l2.M_5_50"         "10.l2.ldscore.gz"   
 [4] "10.l2.M_5_50"        "11.l2.ldscore.gz"    "11.l2.M_5_50"       
 [7] "12.l2.ldscore.gz"    "12.l2.M_5_50"        "13.l2.ldscore.gz"   
[10] "13.l2.M_5_50"        "14.l2.ldscore.gz"    "14.l2.M_5_50"       
[13] "15.l2.ldscore.gz"    "15.l2.M_5_50"        "16.l2.ldscore.gz"   
[16] "16.l2.M_5_50"        "17.l2.ldscore.gz"    "17.l2.M_5_50"       
[19] "18.l2.ldscore.gz"    "18.l2.M_5_50"        "19.l2.ldscore.gz"   
[22] "19.l2.M_5_50"        "2.l2.ldscore.gz"     "2.l2.M_5_50"        
[25] "20.l2.ldscore.gz"    "20.l2.M_5_50"        "21.l2.ldscore.gz"   
[28] "21.l2.M_5_50"        "22.l2.ldscore.gz"    "22.l2.M_5_50"       
[31] "3.l2.ldscore.gz"     "3.l2.M_5_50"         "4.l2.ldscore.gz"    
[34] "4.l2.M_5_50"         "5.l2.ldscore.gz"     "5.l2.M_5_50"        
[37] "6_old.l2.ldscore.gz" "6_old.l2.M_5_50"     "6.l2.ldscore.gz"    
[40] "6.l2.M_5_50"         "7.l2.ldscore.gz"     "7.l2.M_5_50"        
[43] "

**Why HapMap 3 SNPs?**
For LDSC to work as expected, the included SNPs must undergo rigorous Quality Control (QC). To ensure this, LDSC is typically performed using only HM3 SNPs because:
* HM3 SNPs are expected to be well-imputed across most GWAS cohorts.
* This reduces noise from poorly imputed or ultra-rare variants.


Let's inspect the HM3 SNP list that we will pass to `munge()`.


### ❓ Question 1: HapMap 3 SNPs
How many SNPs are there in the HapMap3 reference file?

<details>
<summary>💡 Click here to reveal the answer!</summary>

There are **1,217,311** SNPs included in the HM3 reference file.

*(Note: The first line is the header, so the true count is `wc -l` minus 1).*
</details>


In [8]:
%%R
hm3_file <- file.path(ref_data_dir, "w_hm3.snplist")

# Take a look at the first few lines
cat(system(paste("head", hm3_file), intern = TRUE), sep = "\n")

# Count the number of lines
cat(system(paste("wc -l", hm3_file), intern = TRUE), sep = "\n")


SNP	A1	A2
rs3094315	G	A
rs3131972	A	G
rs3131969	A	G
rs1048488	C	T
rs3115850	T	C
rs2286139	C	T
rs12562034	A	G
rs4040617	G	A
rs2980300	T	C
1217312 EUR/eur_w_ld_chr//w_hm3.snplist


We will restrict the LDSC analyses to the HM3 SNPs in this practical because we only have the pre-computed LD scores for these SNPs. Alternatively, one could compute LD scores for all QC-ed SNPs in a reference panel, but that requires more extensive computation!


### 🛡️ QC Filters for LDSC

We can also specify additional QC filters during munging.
For example, we might drop SNPs with imputation $R^2 < 0.90$ or $\text{MAF} < 0.01$.

*(Note: These filters will only be applied if the GWAS sumstats file includes `INFO` and `MAF` columns).*


In [9]:
%%R
info_cutoff <- 0.9
maf_cutoff  <- 0.01


---
## 🧹 4. Munging Summary Statistics: BMI Example

We will perform the analysis in two steps.
First, we use **`munge()`** to "clean" your GWAS sumstats file by selecting and renaming the required columns. It also filters out variants not present in the reference list, removes non-biallelic variants, and drops variants based on MAF and INFO cutoffs.

For this tutorial, we will practice `munge()` using only **Chromosome 22** data for Body Mass Index (BMI).


### Inspecting the Raw Data
LDSC requires specific column names in the munged data:
* `SNP` - SNP identifier (e.g., rsID).
* `N` - Sample size.
* `Z` - Z-score (or alternatively, a P-value and a signed statistic like `BETA` or `OR`).
* `A1` - Effect allele.
* `A2` - Other allele.


In [10]:
%%R
bmi_file <- file.path(
  sumstat_dir,
  "CHR22only_Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz"
)
cat(system(paste0("zcat ", bmi_file, " | head"), intern=TRUE), sep="\n")


CHR	POS	SNP	Tested_Allele	Other_Allele	BETA	SE	P	N	MAF
22	28273339	rs1000112	C	G	-0.0051	0.0031	0.097	679794	0.08326
22	36890105	rs1000427	A	G	-0.001	0.0028	0.73	692567	0.1118
22	24026845	rs1000470	A	C	0.0017	0.0029	0.55	675111	0.1074
22	20202729	rs1000539	A	G	-0.0025	0.0019	0.19	669369	0.2933
22	26831077	rs1000815	A	G	-0.0077	0.0017	9.2e-06	692285	0.4701
22	22051709	rs10009	A	G	0.0066	0.0018	0.00018	691323	0.3985
22	26403488	rs1001022	T	C	-0.0088	0.0055	0.11	549516	0.03438
22	34131736	rs1001213	A	G	0.0042	0.0038	0.27	690008	0.05958
22	33763247	rs1001223	T	C	0.005	0.0045	0.26	665582	0.04613


### 🔄 Reformatting the Sumstats
We need to verify which allele is the "effect allele" for which the association statistic `BETA` is signed (positive or negative).
Let's read the data, rename the columns to match what `munge()` expects, and save a clean copy.


In [11]:
%%R
bmi_GWAS <- fread(bmi_file, header = TRUE, data.table = FALSE)
str(bmi_GWAS)


'data.frame':	29991 obs. of  10 variables:
 $ CHR          : int  22 22 22 22 22 22 22 22 22 22 ...
 $ POS          : int  28273339 36890105 24026845 20202729 26831077 22051709 26403488 34131736 33763247 36630949 ...
 $ SNP          : chr  "rs1000112" "rs1000427" "rs1000470" "rs1000539" ...
 $ Tested_Allele: chr  "C" "A" "A" "A" ...
 $ Other_Allele : chr  "G" "G" "C" "G" ...
 $ BETA         : num  -0.0051 -0.001 0.0017 -0.0025 -0.0077 0.0066 -0.0088 0.0042 0.005 -0.004 ...
 $ SE           : num  0.0031 0.0028 0.0029 0.0019 0.0017 0.0018 0.0055 0.0038 0.0045 0.0035 ...
 $ P            : num  9.7e-02 7.3e-01 5.5e-01 1.9e-01 9.2e-06 1.8e-04 1.1e-01 2.7e-01 2.6e-01 2.5e-01 ...
 $ N            : int  679794 692567 675111 669369 692285 691323 549516 690008 665582 677031 ...
 $ MAF          : num  0.0833 0.1118 0.1074 0.2933 0.4701 ...


Look at the columns in the BMI sumstats.

Are the column names informative about the "effect allele"?

In this case, **YES**, as the effect allele is labeled `Tested_Allele`!

This is helpful, as we do not have a README file accompanying the data.

If the column names are uninformative and there is no README file, do not hesitate to verify with the corresponding author!

Now, using the following block of code, we'll rename the columns to what LDSC expects and save the file with the new column names. Note that renaming the columns is not required if the sumstats file has standard column names that the program can interpret correctly - which is actually true here. However, that may not always be the case, so make sure to check.

In [ ]:
%%R
bmi_GWAS <- dplyr::rename(
  bmi_GWAS,
  A1 = Tested_Allele,
  A2 = Other_Allele
)
head(bmi_GWAS)


In [ ]:
%%R
bmi_updated_file <- file.path(sumstat_dir, "BMI_GWAS_for_LDSC.txt")
fwrite(
  bmi_GWAS,
  file = bmi_updated_file,
  sep = "\t",
  col.names = TRUE,
  row.names = TRUE,
  quote = FALSE
)
cat(system(paste("head", bmi_updated_file), intern=TRUE), sep="\n")


> ⚠️ **Note:** Some GWAS sumstats may report the "effect allele frequency" rather than MAF. In that case, we'd need to manually convert the effect allele frequency to MAF before filtering!


### 🏃‍♂️ Running `munge()`
Now, let's run the `munge()` function on our updated BMI sumstats.


In [ ]:
%%R
library(GenomicSEM)
bmi_munged <- file.path(sumstat_dir, "munged_BMI_GWAS_chr22_for_LDSC")

munge(
  files = bmi_updated_file,
  hm3 = hm3_file,
  trait.names = bmi_munged,
  info.filter = info_cutoff,
  maf.filter = maf_cutoff
)


### 🔍 Reviewing the Munge Log
Examine the screen output/log from the `munge()` command and answer the following questions:


### ❓ Question 2: Effect Allele
Which column is interpreted as the effect allele (A1)?

<details>
<summary>💡 Click here to reveal the answer!</summary>

The **`A1`** column!

Remember that we explicitly renamed `Tested_Allele` to `A1` to ensure `GenomicSEM` correctly identified it.
</details>


### ❓ Question 3: Starting SNPs
How many SNPs were present in the raw sumstats file?

<details>
<summary>💡 Click here to reveal the answer!</summary>

There were **29,991 SNPs** present in the raw sumstats file.
</details>


### ❓ Question 4: Filtered by Reference
How many SNPs were dropped because they were not present in the reference HapMap3 file?

<details>
<summary>💡 Click here to reveal the answer!</summary>

**15,608 SNPs** were dropped because they were not in the HM3 reference list.
</details>


### ❓ Question 5: Filtered by INFO
How many SNPs were dropped due to low imputation quality (INFO)?

<details>
<summary>💡 Click here to reveal the answer!</summary>

**None!** There was no `INFO` column in the sumstats, so this filter was not applied.
</details>


### ❓ Question 6: Filtered by MAF
How many SNPs were dropped due to low or missing MAF?

<details>
<summary>💡 Click here to reveal the answer!</summary>

**1 SNP** was dropped due to low or missing MAF.

*(Usually, there will be more, but this file was already pre-filtered prior to the tutorial).*
</details>


### ❓ Question 7: Final SNPs
How many SNPs are left in the munged sumstats file?

<details>
<summary>💡 Click here to reveal the answer!</summary>

**14,382 SNPs** are left in the final munged sumstats file.
</details>


---
## 🧮 5. Running LDSC Heritability Analysis

We are done munging the Chromosome 22 subset!

For the actual heritability analysis, we need genome-wide data. The full BMI summary statistics have already been munged for you (for all chromosomes).


In [ ]:
%%R
raw_bmi    <- file.path(sumstat_dir, "Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz")
munged_bmi <- file.path(sumstat_dir, "Munged_BMI_Meta-analysis_Locke_et_al+UKBiobank_2018_for_LDSC.sumstats.gz")


Let's quickly compare the raw file with the munged file, by looking at the first few lines.


In [ ]:
%%R
cat(system(paste0("zcat ", raw_bmi, " | head"), intern=TRUE), sep='\n')
cat(system(paste0("zcat ", munged_bmi, " | head"), intern=TRUE), sep='\n')

### ❓ Question 8: Munged Columns
What are the columns included in the munged sumstats?

<details>
<summary>💡 Click here to reveal the answer!</summary>

`SNP`, `N`, `Z`, `A1`, `A2`
</details>


### 🚀 Executing `ldsc()`
Now, let's run the LDSC analysis on BMI using the full, genome-wide munged sumstats.


In [ ]:
%%R
bmi_ldsc_out <- file.path(sumstat_dir, "BMI")

ldsc(
  traits = munged_bmi,
  sample.prev = NA,
  population.prev = NA,
  ld = ref_data_dir,
  wld = ref_data_dir,
  trait.names = "BMI",
  ldsc.log = bmi_ldsc_out
)


> ⚠️ **Note:** The `ldsc()` function in `GenomicSEM` is primarily designed for *bivariate* LDSC (genetic correlation), so it may print a warning about needing two or more traits. You can safely ignore that warning for this univariate heritability practical!


### 📊 Interpreting LDSC Output

### ❓ Question 9: Inflation Metrics
What are the Mean $\chi^2$ and the $\lambda_{\text{GC}}$? Do these suggest an inflation of GWAS statistics?

<details>
<summary>💡 Click here to reveal the answer!</summary>

* **Mean $\chi^2$** = 3.9345
* **$\lambda_{\text{GC}}$** = 2.7889

Yes! These values are much greater than 1, indicating massive inflation in the test statistics. The question is: is this inflation due to true polygenicity, or confounding (like population stratification)?
</details>


### ❓ Question 10: Intercept
What is the estimate of the LDSC intercept in this analysis? Does it suggest significant bias in the BMI GWAS results?

<details>
<summary>💡 Click here to reveal the answer!</summary>

* **Intercept** = 1.0202 (SE 0.0277)

As the LDSC intercept is not significantly greater than 1, it suggests that there was no significant confounding (population stratification) in the GWAS. The vast majority of the inflation in the QQ plot (and so $\lambda_{GC}$) is due to true, highly polygenic biological signal!
</details>


### 🧠 Intuition check: $\lambda_{GC}$ / mean $\chi^2$ vs. the intercept

In Question 9 you looked at the mean $\chi^2$ (and $\lambda_{GC}$); in Question 10 you looked at the LDSC intercept. Both can exceed 1. Why is a mean $\chi^2 \gg 1$ *not* automatically a sign of confounding?

<details>
<summary>💡 Click here to reveal the answer!</summary>

$\lambda_{GC}$ and the mean $\chi^2$ summarise the **overall inflation** of the test statistics — but they **can't tell you its source**. A highly polygenic trait genuinely inflates *all* statistics through real signal, so a large mean $\chi^2$ is *expected*, not alarming.

The **LDSC intercept** is what separates the two: it isolates the inflation that is **flat in LD score** (confounding) from the part that **grows with LD score** (polygenic signal). A classic illustration from the original LDSC paper:

| Scenario | $\lambda_{GC}$ | LDSC intercept |
|---|---|---|
| Simulated **polygenic** signal | 1.30 | ~1.02 (no confounding) |
| **Stratification** (UK vs Sweden controls) | 1.30 | ~1.32 (confounded) |

Same $\lambda_{GC}$, completely different story — only the intercept tells them apart. This is the key reason LDSC improved on naïve genomic control.
</details>


### ❓ Question 11: Heritability
What is the estimated SNP heritability ($h^2_{\text{SNP}}$) of BMI?

<details>
<summary>💡 Click here to reveal the answer!</summary>

**$h^2_{\text{SNP}}$ of BMI = 0.2091** (SE 0.0063)

*(Or roughly ~21% of the variance in BMI is explained by these common SNPs).*
</details>


🎉 **Congratulations!** You successfully ran an LDSC analysis of BMI.

*Note: LDSC uses the munged sumstats to estimate the heritability/variance per SNP. The program then uses the estimated number of independent SNPs in the reference panel to extrapolate the total SNP heritability.*


---
## 🌟 6. Bonus (Time Permitting): LDSC with LMM Sumstats

We will now compare our previous results to an LDSC analysis using BMI GWAS sumstats from the **[Pan-UKBB project](https://pan.ukbb.broadinstitute.org/)**.

These analyses were performed using a **Linear Mixed Model (LMM)** (specifically SAIGE), which inherently controls for population stratification and relatedness during the association testing phase.

Let's run LDSC on these LMM-derived summary statistics!


In [ ]:
%%R
munged_lmm_bmi <- file.path(sumstat_dir, "BMI.sumstats.gz")

cat(system(paste0("zcat ", munged_lmm_bmi, " | head"), intern=TRUE), sep='\n')

bmi_lmm_ldsc_out <- file.path(sumstat_dir, "BMI_LMM")

ldsc(
  traits = munged_lmm_bmi,
  sample.prev = NA,
  population.prev = NA,
  ld = ref_data_dir,
  wld = ref_data_dir,
  trait.names = "BMI",
  ldsc.log = bmi_lmm_ldsc_out
)


### ❓ Question 12: LMM Intercept
What is the estimate of the LDSC intercept? Does it suggest significant bias in the BMI GWAS results?

<details>
<summary>💡 Click here to reveal the answer!</summary>

* **Intercept** = 1.2284 (SE 0.057)

However, LDSC analyses of LMM sum stats may yield intercepts $>1$
For traits with high SNP heritability
In large samples (such as the UK Biobank, $N > 400k$)

See [Loh, PR., Kichaev, G., Gazal, S. et al. Mixed-model association for biobank-scale datasets. Nat Genet 50, 906–908 (2018)](https://doi.org/10.1038/s41588-018-0144-6) for further details.
</details>


### 🤔 Discussion on $N_{\text{eff}}$
When looking at LMM results, we may want to consider some additional points:

The **Effective $N$** (sometimes referred to as $N_{\text{eff}}$) is often less than the raw sample size $N$ due to the relatedness between some participants being accounted for by the mixed model.

*Could there be residual confounding? Genetic related matrix used as a random effect in LMMs may not control for environmental confounding between family members.*

However, note that the attenuation ratio is 0.097 (SE = 0.0242), which is totally acceptable.

---
## 🧗 Open-ended challenges

Finished early? These open-ended questions go beyond the basic $h^2$ run — they're the kinds of "gotchas" and extensions you'll meet in real heritability analyses. No single right answer; reveal the hints when you want to compare notes.


### ❓ Open challenge: Observed vs liability scale

For a **binary** disease trait, why isn't the "observed-scale" $h^2$ (from plugging 0/1 case-control labels straight into the model) directly comparable between two studies that used **different case/control proportions**? What do we convert to instead?

<details>
<summary>💡 Click here to reveal the answer!</summary>

Observed-scale $h^2$ depends on the **sample prevalence** $P$ (the fraction of cases in *your* study). Case-control studies deliberately oversample cases, so two studies with different $P$ put their phenotypic variance on different scales — their observed-scale $h^2$ values **aren't comparable** (and aren't biologically interpretable).

The fix is the **liability-threshold model**: assume an unobserved continuous "liability" with a threshold set by the **population prevalence** $K$, and convert to **liability-scale $h^2$**, which is independent of ascertainment. The mapping (Lee et al. 2011) uses both $K$ and $P$:
$$h^2_l = h^2_o \, \frac{K^2(1-K)^2}{P(1-P)\,\phi(t)^2},$$
where $t$ is the liability threshold and $\phi(t)$ the standard-normal density there. Always compare disease heritabilities on the **liability scale**.
</details>


### ❓ Open challenge: What does standard LDSC assume about every SNP?

By default, LDSC assumes that — in expectation — **every SNP explains the same amount of heritability**, regardless of its allele frequency (the "$\alpha = -1$" assumption). Where might this break down, and how could it bias $h^2$?

<details>
<summary>💡 Click here to reveal the answer!</summary>

Standard LDSC models each (standardised) SNP's effect-size variance as $\sigma^2_g / M$ — independent of minor allele frequency (MAF), LD, and function. Equivalently, on the per-allele scale it assumes rarer variants have *larger* effects (the $\alpha = -1$ assumption): a reasonable baseline under negative selection, but **not exact for every trait**.

If the true frequency–effect relationship differs (empirically $\alpha$ often sits around $-0.25$ to $-0.5$), the LD scores you regress on are slightly mis-weighted and $h^2$ can be biased. Methods like **SumHer/LDAK** (LD weights plus a tunable $\alpha$) relax these assumptions. The unifying idea: *each method is just a different choice of what $\mathrm{Var}(b_j)$ should be before the same moment-based regression.*
</details>


### ❓ Open challenge: Is a functional category "punching above its weight"?

Suppose you suspect SNPs in **coding regions** contribute disproportionately to a trait's heritability. How could you use **stratified LDSC (S-LDSC)** to test this?

<details>
<summary>💡 Click here to reveal the answer!</summary>

**S-LDSC** partitions heritability across (overlapping) functional **annotations** instead of estimating one genome-wide $h^2$. You compute a separate LD score per annotation and fit a *multiple* regression of $\chi^2$ on those annotation-specific LD scores, giving a per-SNP contribution $\tau_c$ for each category.

To ask whether coding SNPs punch above their weight, compare the category's **share of heritability** to its **share of SNPs**:
$$\text{enrichment} = \frac{h^2(\mathcal{C}) \,/\, h^2}{|\mathcal{C}| \,/\, M}.$$
Enrichment $> 1$ means the category explains more heritability than its size alone predicts. This is the basis of the Finucane et al. 2015 "baseline model".
</details>


### ❓ Open challenge: Why must the LD scores match your GWAS ancestry?

We used `eur_w_ld_chr/` LD scores for a European GWAS. What would go wrong if you used **European** LD scores to estimate $h^2$ from an **East Asian** GWAS?

<details>
<summary>💡 Click here to reveal the answer!</summary>

LDSC's regression rests on each SNP's **LD score** — how much LD it tags — and **LD patterns differ between ancestries** (haplotype structure and allele frequencies). Using European LD scores for an East Asian GWAS regresses the test statistics on the **wrong** $\ell_j$ values, **biasing the slope** and hence $h^2$.

Rule of thumb: **always use LD scores computed in (or well-matched to) the genetic ancestry of your GWAS sample.** This is the single-trait analogue of why standard LDSC can't do cross-ancestry genetic correlation.
</details>


---
## 🏁 End of Practical 1

You have now completed the Practical 1: SNP heritability esimations using LDSC!
